# GeoCadastral AI — CadastreVision Boundary Segmentation Training Notebook

**Project:** GeoCadastral AI (Smart India Hackathon 2026)

**Task:** Cadastral / Parcel Boundary Delineation from Aerial Imagery

**Dataset Reference:** CadastreVision Benchmark (CadNET)

**Execution Environment:** Google Colab (Cloud GPU - NVIDIA T4 / V100)

---
> [!IMPORTANT]
> **STATUTORY NOTICE:** Outputs generated by this model are AI-assisted decision-support delineations. They do not independently establish legal title and require authoritative survey verification.

## 1. Environment Setup & Dependency Installation

In [ ]:
!pip install -q onnx onnxruntime opencv-python-headless pillow shapely rasterio pyproj matplotlib pyyaml tqdm

import os
import sys
import json
import time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader

REPO_ROOT = Path("/content/SIH-project")
if not REPO_ROOT.exists():
    REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repository root: {REPO_ROOT}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    print("WARNING: Colab GPU is not enabled; training will run on CPU and may be slow.")
else:
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

## 2. Configuration & Hyperparameters

Easily adjust hyperparameters below. Safe defaults are provided for Colab T4 GPUs.

In [ ]:
# Use explicit Colab paths; never assume a Windows workstation dataset.
DATA_ROOT = Path("/content/data/cadastrevision")
if not DATA_ROOT.exists():
    DATA_ROOT = REPO_ROOT / "data" / "cadastrevision"
CHECKPOINT_DIR = REPO_ROOT / "results" / "checkpoints"
EXPORT_ROOT = Path("/content/backend_export")
PATCH_SIZE = 512
OVERLAP_RATIO = 0.15
BATCH_SIZE = 4
EPOCHS = 15
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Dataset root: {DATA_ROOT}")
print(f"Checkpoint directory: {CHECKPOINT_DIR}")
print(f"Export directory: {EXPORT_ROOT}")
print(f"Running on device: {DEVICE} with batch size {BATCH_SIZE}")

## 3. Dataset Acquisition & Inspection (Real CadastreVision Subset)

Place only an authorised, labelled subset at `/content/data/cadastrevision/` or under the repository's `data/cadastrevision/`. The notebook fails if the manifest, images, masks, or independent scenes are missing. It never generates synthetic training data.

In [ ]:
# Load only a real, labelled CadastreVision subset.
manifest_path = DATA_ROOT / "manifest.json"
if not manifest_path.exists():
    raise FileNotFoundError(
        "REAL CADASTREVISION DATA NOT FOUND: place an authorised labelled subset in "
        f"{DATA_ROOT} and provide manifest.json before training."
    )

with manifest_path.open("r", encoding="utf-8") as manifest_file:
    manifest = json.load(manifest_file)

if manifest.get("is_synthetic") or "synthetic" in manifest.get("dataset_name", "").lower():
    raise ValueError("Synthetic fixtures are not permitted for claimed training.")

samples = manifest.get("samples", [])
if not samples:
    raise ValueError("The real-data manifest contains no labelled samples.")

def resolve_sample_path(value):
    path = Path(value)
    if path.is_absolute() and path.exists():
        return str(path)
    candidates = [REPO_ROOT / path, DATA_ROOT / path.name, DATA_ROOT.parent / path]
    for candidate in candidates:
        if candidate.exists():
            return str(candidate)
    raise FileNotFoundError(f"Training file not found: {value}")

for sample in samples:
    sample["image_path"] = resolve_sample_path(sample["image_path"])
    sample["mask_path"] = resolve_sample_path(sample["mask_path"])

scene_count = len({sample.get("scene_id") or sample.get("flight_id") for sample in samples})
if scene_count < 3:
    raise ValueError(f"At least 3 scenes are required for leakage-free splits; found {scene_count}.")

print(f"Loaded {len(samples)} labelled samples across {scene_count} scenes.")

## 4. Leakage-Free Scene Grouping (Train / Val / Test Split)

We group by `scene_id` to strictly prevent spatial data leakage between training and testing sets.

In [ ]:
from ai_training.dataset import split_dataset_by_scene

splits = split_dataset_by_scene(samples, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15)
train_samples = splits["train"]
val_samples = splits["val"]
test_samples = splits["test"]
print(f"Train samples: {len(train_samples)}")
print(f"Val samples:   {len(val_samples)}")
print(f"Test samples:  {len(test_samples)}")
print("Scene-level split verified: no scene is shared between partitions.")

## 5. Ground Truth Mask Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
sample = train_samples[0]

img = Image.open(sample["image_path"])
mask = Image.open(sample["mask_path"])

axes[0].imshow(img)
axes[0].set_title("Aerial Image (512x512)")
axes[0].axis("off")

axes[1].imshow(mask, cmap="gray")
axes[1].set_title("Cadastral Boundary Ground Truth")
axes[1].axis("off")

axes[2].imshow(img)
axes[2].imshow(mask, cmap="spring", alpha=0.5)
axes[2].set_title("Overlay Verification")
axes[2].axis("off")

plt.tight_layout()
plt.show()

## 6. PyTorch Dataset & DataLoader

In [ ]:
class CadastralColabDataset(Dataset):
    def __init__(self, samples, augment=False):
        self.samples = samples
        self.augment = augment
        self.mean = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(1, 1, 3)
        self.std = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(1, 1, 3)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        img = np.array(Image.open(s["image_path"]).convert("RGB"), dtype=np.float32) / 255.0
        mask = (np.array(Image.open(s["mask_path"]).convert("L"), dtype=np.float32) > 127).astype(np.float32)

        if self.augment:
            if np.random.rand() > 0.5:
                img = np.fliplr(img).copy()
                mask = np.fliplr(mask).copy()
            if np.random.rand() > 0.5:
                img = np.flipud(img).copy()
                mask = np.flipud(mask).copy()

        img = (img - self.mean) / self.std
        img_tensor = torch.from_numpy(np.transpose(img, (2, 0, 1)).astype(np.float32))
        mask_tensor = torch.from_numpy(np.expand_dims(mask, 0).astype(np.float32))
        return img_tensor, mask_tensor

train_loader = DataLoader(CadastralColabDataset(train_samples, augment=True), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(CadastralColabDataset(val_samples, augment=False), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(CadastralColabDataset(test_samples, augment=False), batch_size=1, shuffle=False)
print("DataLoaders successfully configured.")

## 7. Model Architecture (CadastralBoundaryUNet) & Loss Function

In [ ]:
from ai_training.model import CadastralBoundaryUNet
from ai_training.losses import BCEDiceLoss

model = CadastralBoundaryUNet(in_channels=3, num_classes=1, base_channels=32).to(DEVICE)
criterion = BCEDiceLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
print(f"Model initialized with {sum(p.numel() for p in model.parameters()):,} parameters.")

## 8. Real Training & Validation Loop

In [ ]:
history = []
best_val_loss = float("inf")

print("Starting training loop...")
for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        preds = model(imgs)
        loss = criterion(preds, masks)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item() * imgs.size(0)

    model.eval()
    val_loss = 0.0
    val_tp, val_fp, val_fn = 0, 0, 0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            preds = model(imgs)
            val_loss += criterion(preds, masks).item() * imgs.size(0)
            p_bin = (preds.cpu().numpy() >= 0.45).astype(np.uint8)
            m_bin = (masks.cpu().numpy() > 0.5).astype(np.uint8)
            val_tp += int(np.sum((m_bin == 1) & (p_bin == 1)))
            val_fp += int(np.sum((m_bin == 0) & (p_bin == 1)))
            val_fn += int(np.sum((m_bin == 1) & (p_bin == 0)))

    train_loss /= max(1, len(train_loader.dataset))
    val_loss /= max(1, len(val_loader.dataset))
    val_iou = val_tp / (val_tp + val_fp + val_fn + 1e-7)
    val_f1 = (2.0 * val_tp) / (2.0 * val_tp + val_fp + val_fn + 1e-7)
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "val_iou": val_iou, "val_f1": val_f1})
    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val IoU: {val_iou:.4f} | Val F1: {val_f1:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            "model_state_dict": model.state_dict(),
            "metrics": {"val_iou": val_iou, "val_f1": val_f1},
            "epoch": epoch,
            "training_provenance": {
                "dataset_name": manifest["dataset_name"],
                "is_synthetic": False,
                "sample_count": len(samples),
                "scene_count": scene_count
            }
        }, CHECKPOINT_DIR / "cadastral_boundary_best.pt")

print(f"Best model checkpoint saved with Val Loss: {best_val_loss:.4f}")

## 9. Plot Loss & Validation Metrics

In [ ]:
epochs_range = [h["epoch"] for h in history]
train_losses = [h["train_loss"] for h in history]
val_losses = [h["val_loss"] for h in history]
val_ious = [h["val_iou"] for h in history]

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, train_losses, label="Train Loss", color="#3B82F6")
plt.plot(epochs_range, val_losses, label="Val Loss", color="#EF4444")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(epochs_range, val_ious, label="Val Boundary IoU", color="#10B981")
plt.xlabel("Epoch")
plt.ylabel("IoU")
plt.title("Validation Boundary IoU")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Test Set Evaluation (Real Measured Metrics)

In [ ]:
model.load_state_dict(torch.load(CHECKPOINT_DIR / "cadastral_boundary_best.pt", map_location=DEVICE)["model_state_dict"])
model.eval()

test_tp, test_fp, test_fn = 0, 0, 0
test_predictions = []

with torch.no_grad():
    for imgs, masks in test_loader:
        imgs = imgs.to(DEVICE)
        preds = model(imgs)
        p_bin = (preds.cpu().numpy()[0, 0] >= 0.45).astype(np.uint8)
        m_bin = (masks.numpy()[0, 0] > 0.5).astype(np.uint8)
        test_tp += int(np.sum((m_bin == 1) & (p_bin == 1)))
        test_fp += int(np.sum((m_bin == 0) & (p_bin == 1)))
        test_fn += int(np.sum((m_bin == 1) & (p_bin == 0)))
        test_predictions.append((imgs.cpu().numpy()[0], m_bin, preds.cpu().numpy()[0, 0]))

test_iou = test_tp / (test_tp + test_fp + test_fn + 1e-7)
test_f1 = (2.0 * test_tp) / (2.0 * test_tp + test_fp + test_fn + 1e-7)
test_prec = test_tp / (test_tp + test_fp + 1e-7)
test_rec = test_tp / (test_tp + test_fn + 1e-7)

print("FINAL MEASURED TEST METRICS")
print(f"Boundary IoU: {test_iou:.4f}")
print(f"Boundary F1: {test_f1:.4f}")
print(f"Precision: {test_prec:.4f}")
print(f"Recall: {test_rec:.4f}")

## 11. Visual Inspection (Image, Ground Truth, Predicted Probability)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
t_img_norm, t_gt, t_pred = test_predictions[0]

# Unnormalize image
mean = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1)
std = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1)
t_img = np.clip(np.transpose(t_img_norm * std + mean, (1, 2, 0)), 0, 1)

axes[0].imshow(t_img)
axes[0].set_title("Test Aerial Image (Scene 3)")
axes[0].axis("off")

axes[1].imshow(t_gt, cmap="gray")
axes[1].set_title("Ground Truth Boundary Mask")
axes[1].axis("off")

im = axes[2].imshow(t_pred, cmap="hot")
axes[2].set_title("AI Boundary Probability Map")
axes[2].axis("off")
plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

## 12. ONNX Model Export & Verification

This cell is **fully self-contained**. It re-creates the model architecture, loads the trained checkpoint, and exports to ONNX. It will work even if you restart the Colab runtime.

In [ ]:
# Export only the checkpoint produced by the real training cells.
from ai_training.export_onnx import export_pytorch_to_onnx

checkpoint_path = CHECKPOINT_DIR / "cadastral_boundary_best.pt"
if not checkpoint_path.exists():
    raise FileNotFoundError("No trained checkpoint found. Complete a real training run before exporting ONNX.")

onnx_path = EXPORT_ROOT / "geocadastral_cadastral_boundary_v1.onnx"
metadata_path = EXPORT_ROOT / "geocadastral_cadastral_boundary_v1_meta.json"
metadata = export_pytorch_to_onnx(
    checkpoint_path=str(checkpoint_path),
    output_onnx_path=str(onnx_path),
    metadata_path=str(metadata_path),
    test_metrics={
        "test_iou": float(test_iou),
        "test_f1": float(test_f1),
        "test_precision": float(test_prec),
        "test_recall": float(test_rec)
    }
)

if metadata["status"] != "TRAINED_AND_VERIFIED":
    raise RuntimeError(f"ONNX verification failed: {metadata['onnx_verification']}")
print("TRAINED_AND_VERIFIED")
print(f"Exported package: {onnx_path}, {metadata_path}")

## 13. Download Files to Your Computer

In [ ]:
from google.colab import files
import shutil

package_root = EXPORT_ROOT / "geocadastral_cadastral_boundary_v1"
package_root.mkdir(parents=True, exist_ok=True)
for source in [onnx_path, metadata_path, checkpoint_path]:
    shutil.copy2(source, package_root / Path(source).name)
package_archive = shutil.make_archive(str(EXPORT_ROOT / "geocadastral_cadastral_boundary_v1"), "zip", package_root)
files.download(package_archive)
print(f"Downloaded model package: {package_archive}")
print("Install the ONNX and metadata files in backend/models/ only after reviewing TRAINED_AND_VERIFIED status.")

## 14. Summary

After running all cells above, you should have:

| File | Purpose |
|------|----------|
| `/content/backend_export/geocadastral_cadastral_boundary_v1.onnx` | ONNX model exported from the real trained checkpoint |
| `/content/backend_export/geocadastral_cadastral_boundary_v1_meta.json` | Metadata with measured test metrics and PyTorch/ONNX verification |
| `results/checkpoints/cadastral_boundary_best.pt` | PyTorch checkpoint from the real training run |
| `/content/backend_export/geocadastral_cadastral_boundary_v1.zip` | Downloadable model package |

Copy the `.onnx` and `_meta.json` files into the repository's `backend/models/` folder only after reviewing the metadata status `TRAINED_AND_VERIFIED`.
GeoCadastral AI's `ModelManager` loads a verified ONNX model for tiled inference; otherwise it reports the heuristic development fallback.

All output remains preliminary decision-support and requires authorised survey/GNSS/human validation.